# Unsupervised GraphSAGE via Aura Session (typed endpoints)

Train an unsupervised GraphSAGE model on Cora and compute node embeddings, using the typed
`gds.graph_sage.unsupervised` endpoints introduced in graphdatascience-client PR #1289
(instead of raw Arrow `JobClient` calls).

In [ ]:
%pip install "graphdatascience>=2.0a1" python-dotenv

## Configuration

In [ ]:
SESSION_NAME = "graphsage-unsup-via-session-typed"
CLOUD_PROVIDER, CLOUD_REGION = "gcp", "us-east4"

## Credentials

In [ ]:
import os

from dotenv import load_dotenv

# This allows to load required secrets from `.env` file in local directory
load_dotenv(".env")

## Get or Create Session

In [ ]:
from datetime import timedelta

from graphdatascience.session import AuraAPICredentials, CloudLocation, GdsSessions, SessionMemory

api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    project_id=os.environ["PROJECT_ID"],
)

sessions = GdsSessions(api_credentials=api_credentials)

gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=SessionMemory.m_4GB,
    ttl=timedelta(minutes=30),
    gpu=True,
    cloud_location=CloudLocation(CLOUD_PROVIDER, CLOUD_REGION),
)

gds.verify_connectivity()

## Build Catalog Arrow Endpoints

In [ ]:
from graphdatascience.procedure_surface.arrow.catalog.catalog_arrow_endpoints import CatalogArrowEndpoints

arrow_client = gds._authenticated_arrow_client
catalog_arrow_endpoints = CatalogArrowEndpoints(arrow_client)

## Load Cora Dataset

Cora `Paper` nodes carry a numeric `features` vector and are connected by `CITES` relationships.
We load it undirected so the random walks used by unsupervised GraphSAGE can traverse in both directions.

In [ ]:
_ = catalog_arrow_endpoints.drop("cora", fail_if_missing=False)
G = catalog_arrow_endpoints.datasets.load_cora(undirected=True)
print("Loaded graph:", G.name())

## Train GraphSAGE (Unsupervised)

`gds.graph_sage.unsupervised.train` submits the training job, blocks until completion (with progress
reporting), and returns the trained model together with a train summary result.
`featureProperties` selects the node properties used as input features (here Cora's `features` vector).
`nodeLabels` and `relationshipTypes` default to `*` (project everything).

In [ ]:
modelname = "gs_cora_unsup_typed"

In [ ]:
model, train_result = gds.graph_sage.unsupervised.train(
    G,
    model_name=modelname,
    feature_properties=["features"],
    epochs=10,
    embedding_dimension=64,
    learning_rate=0.001,
    num_neighbors=[25, 10],
    activation_function="relu",
    aggregator="mean",
    num_walks=10,
    walk_depth=3,
    random_seed=42,
)

print("Trained model:", model.name())

In [ ]:
train_result

## Run Inference

`model.predict_stream` runs the predict job against the stored model and streams the results.
Unsupervised GraphSAGE produces a node embedding per node (rather than class predictions).
Inference projects the same `featureProperties` the model was trained on.

In [ ]:
embeddings_df = model.predict_stream(
    G,
    feature_properties=["features"],
    random_seed=42,
)

print("Columns:", list(embeddings_df.columns))
embeddings_df

## Results

In [ ]:
print(f"Computed embeddings for {len(embeddings_df)} nodes")
embeddings_df

## (Optional) cleanup

In [ ]:
gds.model.delete(modelname, fail_if_missing=False)
sessions.delete(session_name=SESSION_NAME)